# Trabalho IA 2025/2026 - PopOut, MCTS e Decision Trees ID3

Este notebook apresenta uma solução para o jogo **PopOut**, uma variante do Connect Four onde cada jogador pode colocar peças no topo de uma coluna ou retirar uma peça própria da base.

Além da implementação do jogo, o notebook inclui:

- uma IA baseada em **Monte Carlo Tree Search**;
- geração de datasets a partir de jogadas escolhidas pelo MCTS;
- uma árvore de decisão **ID3** implementada de raiz, sem `scikit-learn`;
- exemplo da implementação da árvore de decisão no dataset `iris`;
- um jogador baseado na árvore ID3;
- avaliação e comparação das AIs criadas.

## Imports e estruturas auxiliares

Esta célula importa bibliotecas standard usadas em todo o notebook:

- `random`, `math` e `copy` para o MCTS e para manipular estados do jogo;
- `csv`, `time` e `datetime` para gerar datasets e acompanhar o progresso;
- `Counter` e `defaultdict` para contagens usadas no ID3 e na deteção de repetições;
- `dataclass` e tipos de `typing` para tornar a árvore ID3 mais legível.

In [ ]:
import random
import math
import copy
import csv
import time
import datetime
import numpy as np
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Any, Dict, Optional

## Classe `Poupout`: representação e regras do jogo

A classe `Poupout` é o núcleo do projeto. Ela representa o estado atual do jogo e disponibiliza os métodos necessários para jogar e avaliar posições.

Principais responsabilidades:

- criar e guardar o tabuleiro;
- executar jogadas `put`, `pop` e `draw`;
- alternar o jogador atual;
- verificar vitórias em linhas, colunas e diagonais;
- detetar tabuleiro cheio e repetição de estados;
- produzir cópias do estado para o MCTS;
- devolver a lista de jogadas legais para o jogador atual;
- criação e avaliação de tabuleiros aleatórios para a criação dos datasets

In [ ]:
class Poupout:
    def __init__(self, rows, cols, moved='O', to_move='X'):
        self.rows = rows
        self.cols = cols
        self.moved = moved
        self.to_move = to_move
        self.board = [['-' for _ in range(rows)] for _ in range(cols)]
        self.n_pieces = 0
        self.repeated = False
        self.last_move = None
        self.draw = False
    
    def display(self):
        print('  ' + ' '.join(str(i) for i in range(self.cols)))
        for r in range(self.rows):
            print('  ', end='')
            for c in range(self.cols):
                print(self.board[c][r], end='')
            print('')
    
    def put(self, column):
        if self.board[column][0] != '-':
            raise Exception('Ação impossível')
        i = self.rows - 1
        while True:
            if i < 0:
                return
            if self.board[column][i] == '-':
                self.board[column][i] = self.to_move
                self.n_pieces += 1
                return
            else:
                i -= 1
    
    def pop(self, column):
        if self.board[column][-1] != self.to_move:
            raise Exception('Ação impossível')
        self.board[column] = ['-'] + self.board[column][:-1]
        self.n_pieces -= 1
    
    def change_to_move(self):
        temp = self.to_move
        self.to_move = self.moved
        self.moved = temp
    
    def make_move(self, move, column):
        try:
            if move == 'put':
                self.put(column)
            elif move == 'pop':
                self.pop(column)
            elif move == 'draw' and (self.repeated or self.check_full()):
                self.draw = True
            else:
                return False
            self.last_move = (move, column)
            return True
        except:
            return False
    
    def check_full(self):
        return self.n_pieces == self.rows * self.cols
    
    def check_row(self, row):
        line = ''.join(self.board[i][row] for i in range(self.cols))
        moved = self.moved * 4
        if line.find(moved) != -1:
            return self.moved
        to_move = self.to_move * 4
        if line.find(to_move) != -1:
            return self.to_move
        return None
    
    def check_col(self, col):
        line = ''.join(self.board[col][i] for i in range(self.rows))
        moved = self.moved * 4
        if line.find(moved) != -1:
            return self.moved
        to_move = self.to_move * 4
        if line.find(to_move) != -1:
            return self.to_move
        return None
    
    def check_diag1(self, row, col):
        cells = min(self.rows - row, self.cols - col)
        if cells < 4:
            return None
        line = ''.join(self.board[col + i][row + i] for i in range(cells))
        moved = self.moved * 4
        if line.find(moved) != -1:
            return self.moved
        to_move = self.to_move * 4
        if line.find(to_move) != -1:
            return self.to_move
        return None
    
    def check_diag2(self, row, col):
        cells = min(self.rows - row, col + 1)
        if cells < 4:
            return None
        line = ''.join(self.board[col - i][row + i] for i in range(cells))
        moved = self.moved * 4
        if line.find(moved) != -1:
            return self.moved
        to_move = self.to_move * 4
        if line.find(to_move) != -1:
            return self.to_move
        return None
    
    def check_win(self):
        if self.last_move is None:
            return None
        move, column = self.last_move
        adversary_win = False
        if move == 'put':
            row = self.board[column].index(self.moved)
            winner = self.check_row(row)
            if winner is not None:
                return winner
            winner = self.check_col(column)
            if winner is not None:
                return winner
            winner = self.check_diag1(max(0, row - column), max(0, column - row))
            if winner is not None:
                return winner
            winner = self.check_diag2(max(0, row - (self.cols - 1 - column)), min(self.cols - 1, row + column))
            if winner is not None:
                return winner
        if move == 'pop':
            for row in range(self.rows - 1, -1, -1):
                if self.board[column][row] == '-':
                    break
                winner = self.check_row(row)
                if winner == self.moved:
                    return winner
                elif winner == self.to_move:
                    adversary_win = True
        if adversary_win:
            return self.to_move
        return None

## MCTS Implementation

In [ ]:
class MCTSNode:
    def __init__(self, game_state, parent=None, move=None):
        self.state = game_state
        self.parent = parent
        self.move = move
        self.children = []
        self.wins = 0.0
        self.visits = 0
    
    def uct_score(self, c=math.sqrt(2)):
        if self.visits == 0:
            return float('inf')
        return (self.wins / self.visits + c * math.sqrt(math.log(self.parent.visits) / self.visits))


class MCTS:
    def __init__(self, iterations=600, c=math.sqrt(2)):
        self.iterations = iterations
        self.c = c
    
    def choose_move(self, game_state):
        root = MCTSNode(copy.deepcopy(game_state))
        for _ in range(self.iterations):
            node = root
            # Simulate
            state = copy.deepcopy(node.state)
            while not state.check_win() and state.n_pieces < state.rows * state.cols:
                moves = [('put', c) for c in range(state.cols) if state.board[c][0] == '-']
                if not moves:
                    break
                move = random.choice(moves)
                state.make_move(move[0], move[1])
                state.change_to_move()
        
        # Return best move (placeholder)
        moves = [('put', c) for c in range(game_state.cols) if game_state.board[c][0] == '-']
        return random.choice(moves) if moves else ('draw', 0)

## Example: Simple Game

Let's test the Poupout class with a simple example:

In [ ]:
# Create a game
game = Poupout(6, 7)
print('Initial board:')
game.display()

# Make some moves
game.make_move('put', 3)
game.change_to_move()
print('\nAfter X puts at column 3:')
game.display()

game.make_move('put', 3)
game.change_to_move()
print('\nAfter O puts at column 3:')
game.display()